# 04 Size Stroke and Targeted Augmentation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP007_KNN_Normal_Augmentation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP008_KNN_Normal_Aug_Ratio.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP009_KNN_Normal_Stroke_Strength.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP014_KNN_Normal_Targeted_Augmentation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b63e3ba53fb9e43a06e8661e9063d0d754a67730'}, '05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '22840a56aa789dd0b417bd191accb679f11b8c7e'}, '05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'c50900f37f3a50994dedc8edd6906b685db5907f'}, '05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '705f71b31cb9a0e814cd254d1e67642f66a7d9e9'}}

## EXP007_KNN_Normal_Augmentation

원본: [05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py)  
Git blob: `b63e3ba53fb9e43a06e8661e9063d0d754a67730`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-007: KNN Normal, train-only size and stroke augmentation.

Use beside EXP004_KNN_Normal_CoM.py and EXP006_KNN_Normal_BB_Size_fixed.py.
Run from repository root:
    python 05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py

Control: EXP-005 CoM + KNN(k=3, weights=distance, Euclidean).
Variants append exactly 20% of the official Training fold as augmented examples.
Hand-made is development evaluation only, never a source of augmentation.
Original Test and final held-out Te



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-007: KNN Normal, train-only size and stroke augmentation.

Use beside EXP004_KNN_Normal_CoM.py and EXP006_KNN_Normal_BB_Size_fixed.py.
Run from repository root:
    python 05_model_optimize/KNN/Normal/scripts/EXP007_KNN_Normal_Augmentation.py

Control: EXP-005 CoM + KNN(k=3, weights=distance, Euclidean).
Variants append exactly 20% of the official Training fold as augmented examples.
Hand-made is development evaluation only, never a source of augmentation.
Original Test and final held-out Test are not loaded.
"""
from __future__ import annotations

import csv
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import scipy
import sklearn
from scipy.ndimage import grey_dilation, grey_erosion, label, binary_fill_holes
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import LABELS, fixed_split, pipeline, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP006_KNN_Normal_BB_Size_fixed import apply_size, geometry
from Automatic_Centering_Transformer import test_centering

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP007_augmentation'
SEED = 20261121
AUG_FRACTION = 0.20
TARGET_LONG_SIDES = (16, 18, 22, 24)
STROKE_STRENGTH = 0.28
THRESHOLD_TOPOLOGY = 128
OFFICIAL_GUARDRAIL = 0.9569791666666667  # same as EXP-005/006
EXPECTED_OFFICIAL_CORRECT = 18529  # 0.96505208 * 19200
EXPECTED_HANDMADE_CORRECT = 2017  # 0.787890625 * 2560

# 4-connectivity cross, matching cv2.getStructuringElement(MORPH_CROSS, (3,3))
CROSS = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
STRUCTURE8 = np.ones((3, 3), dtype=bool)


def save_csv(path, rows):
    if not rows:
        raise ValueError(f'Empty CSV: {path}')
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def save_cm(path, matrix):
    save_csv(path, [
        {'true/pred': LABELS[i], **{LABELS[j]: int(matrix[i][j]) for j in range(16)}}
        for i in range(16)
    ])


def sample_balanced(y, fraction=AUG_FRACTION, seed=SEED):
    """Class-stratified training indices; no validation/Hand-made sampling."""
    rng = np.random.default_rng(seed)
    groups = []
    for c in range(16):
        ids = np.flatnonzero(y == c)
        n = int(round(len(ids) * fraction))
        if n <= 0 or n >= len(ids):
            raise ValueError(f'Invalid augmentation count for class {c}')
        groups.append(np.sort(rng.choice(ids, size=n, replace=False)))
    selected = np.sort(np.concatenate(groups))
    assert len(np.unique(selected)) == len(selected)
    return selected


def topology_counts(image):
    """Number of connected foreground components and enclosed holes."""
    mask = np.asarray(image) >= THRESHOLD_TOPOLOGY
    ncomp = int(label(mask, structure=STRUCTURE8)[1])
    holes = binary_fill_holes(mask) & ~mask
    nholes = int(label(holes, structure=STRUCTURE8)[1])
    return ncomp, nholes


def stroke_jitter(batch, strength=STROKE_STRENGTH):
    """MLP-style gray morphology blend, cross kernel, topology guarded.

    Deterministic alternation: even images erode, odd images dilate.
    Unlike random pixel noise, this directly tests stroke thickness.
    """
    src = np.asarray(batch, dtype=np.float32).reshape(-1, 28, 28)
    result = np.empty_like(src)
    accepted = rejected = 0
    for i, img in enumerate(src):
        target = (grey_erosion(img, footprint=CROSS, mode='constant', cval=0)
                  if i % 2 == 0 else
                  grey_dilation(img, footprint=CROSS, mode='constant', cval=0))
        proposed = np.clip((1 - strength) * img + strength * target, 0, 255)
        if topology_counts(proposed) == topology_counts(img):
            result[i] = proposed
            accepted += 1
        else:
            result[i] = img
            rejected += 1
    return result.reshape(-1, 784), {'changed': accepted, 'topology_rejected': rejected}


def size_jitter(x, selected, seed=SEED):
    """Balanced across sizes within each class, with no fixed global target."""
    chosen = np.asarray(x[selected], dtype=np.float32)
    out = np.empty_like(chosen)
    rng = np.random.default_rng(seed + 17)
    targets = np.tile(np.asarray(TARGET_LONG_SIDES, dtype=np.int32),
                      (len(selected) + len(TARGET_LONG_SIDES)-1) // len(TARGET_LONG_SIDES))[:len(selected)]
    rng.shuffle(targets)
    for t in TARGET_LONG_SIDES:
        mask = targets == t
        if np.any(mask):
            out[mask], _ = apply_size(chosen[mask], int(t))
    return out, {str(t): int(np.sum(targets == t)) for t in TARGET_LONG_SIDES}


def test_augmentation():
    test_centering()
    x = np.zeros((2, 28, 28), dtype=np.float32)
    x[0, 7:21, 10:18] = 255
    x[1, 6:22, 8:20] = 255
    test_scaled, _ = apply_size(x, 18)
    assert test_scaled.shape == (2, 784)
    assert test_scaled.min() >= 0 and test_scaled.max() <= 255
    altered, record = stroke_jitter(x, strength=STROKE_STRENGTH)
    assert altered.shape == (2, 784)
    assert record['changed'] + record['topology_rejected'] == 2
    assert np.all(np.isfinite(altered)) and 0 <= altered.min() <= altered.max() <= 255
    assert len(geometry(x)[0]) == 2
    print('AUGMENTATION SELF-TEST PASSED: size, stroke topology, range', flush=True)


def knn_model():
    return pipeline(KNeighborsClassifier(n_neighbors=3, weights='distance',
                                         metric='minkowski', p=2), True)


def main():
    test_augmentation()
    xtr, ytr, xva, yva, dataset_sha = fixed_split()
    xhm, yhm, writers, indices = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise RuntimeError('Dataset sizes changed; refusing to compare')
    print(f'Official train/val={len(ytr)}/{len(yva)}; Hand-made={len(yhm)}', flush=True)

    com = BatchedCoM()
    xtr_com = com.transform(xtr)
    xva_com = com.transform(xva)
    xhm_com = com.transform(xhm)

    selected = sample_balanced(ytr)
    print(f'Augmented samples: {len(selected)} extra (original official Training remains intact)', flush=True)
    size_aug, size_counts = size_jitter(xtr, selected)
    stroke_aug, stroke_stats = stroke_jitter(xtr[selected])
    both_aug, both_stats = stroke_jitter(size_aug)
    augmentations = {
        'control_com': None,
        'size_only': size_aug,
        'stroke_only': stroke_aug,
        'size_plus_stroke': both_aug,
    }
    # All augmented examples, and all evaluation samples, receive the same CoM rule.
    rows = []
    writer_rows = []
    class_rows = []
    predict_rows = []
    records = {}
    predictions = {}
    baseline_official = baseline_handmade = None

    for variant, aug in augmentations.items():
        print(f'\n[{variant}] Preparing official Training examples...', flush=True)
        if aug is None:
            train = xtr_com
            target = ytr
        else:
            aug_com = com.transform(aug)
            train = np.concatenate([xtr_com, aug_com], axis=0)
            target = np.concatenate([ytr, ytr[selected]])
        print(f'  Fit rows={len(target)}; augmented={len(target)-len(ytr)}', flush=True)
        model = knn_model()
        start = time.perf_counter()
        model.fit(train, target)
        fit_time = time.perf_counter() - start
        start = time.perf_counter()
        off_pred = np.asarray(model.predict(xva_com), dtype=np.int64)
        off_predict_seconds = time.perf_counter() - start
        start = time.perf_counter()
        hm_pred = np.asarray(model.predict(xhm_com), dtype=np.int64)
        hm_predict_seconds = time.perf_counter() - start
        off = scores(yva, off_pred)
        hm = scores(yhm, hm_pred)
        if variant == 'control_com':
            if (int(np.count_nonzero(off_pred == yva)) != EXPECTED_OFFICIAL_CORRECT or
                    int(np.count_nonzero(hm_pred == yhm)) != EXPECTED_HANDMADE_CORRECT):
                raise RuntimeError('EXP-005 baseline mismatch: stop and check dependencies/data')
            baseline_official, baseline_handmade = off_pred.copy(), hm_pred.copy()
            print('  EXP-005 control reproduced exactly (correct counts)', flush=True)
        records[variant] = {'official': off, 'handmade': hm,
                            'fit_seconds': fit_time, 'official_predict_seconds': off_predict_seconds,
                            'handmade_predict_seconds': hm_predict_seconds}
        predictions[variant] = {'official': off_pred, 'handmade': hm_pred}
        guard = off['accuracy'] >= OFFICIAL_GUARDRAIL
        rows.append({
            'variant': variant, 'training_n': len(target), 'augmented_n': len(target) - len(ytr),
            'official_accuracy': off['accuracy'], 'official_macro_f1': off['macro_f1'],
            'handmade_accuracy': hm['accuracy'], 'handmade_macro_f1': hm['macro_f1'],
            'official_delta_pp_vs_com': 100 * (off['accuracy'] - EXPECTED_OFFICIAL_CORRECT / len(yva)),
            'handmade_delta_pp_vs_com': 100 * (hm['accuracy'] - EXPECTED_HANDMADE_CORRECT / len(yhm)),
            'official_guardrail_pass': int(guard),
            'fit_seconds': fit_time, 'official_predict_seconds': off_predict_seconds,
            'handmade_predict_seconds': hm_predict_seconds,
        })
        for w in (41, 42, 43, 44):
            mask = writers == w
            ss = scores(yhm[mask], hm_pred[mask])
            writer_rows.append({'variant': variant, 'writer': w, 'n': int(mask.sum()),
                                'accuracy': ss['accuracy'], 'macro_f1': ss['macro_f1']})
        for klass in LABELS:
            item = hm['per_class'][klass]
            class_rows.append({'variant': variant, 'class': klass, 'n': item['support'],
                               'precision': item['precision'], 'recall': item['recall'], 'f1': item['f1']})
        print(f"  Official={off['accuracy']:.6f} F1={off['macro_f1']:.6f} | "
              f"Hand-made={hm['accuracy']:.6f} F1={hm['macro_f1']:.6f} | "
              f"guardrail={'PASS' if guard else 'FAIL'}", flush=True)
        del model, train, target

    valid = [r for r in rows if r['official_guardrail_pass']]
    selected_candidate = max(valid, key=lambda r: (r['handmade_macro_f1'], r['handmade_accuracy'],
                                                   r['official_macro_f1'])) if valid else None

    OUTPUT.mkdir(parents=True, exist_ok=True)
    save_csv(OUTPUT / 'candidate_results.csv', rows)
    save_csv(OUTPUT / 'by_writer.csv', writer_rows)
    save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
    for variant in augmentations:
        save_cm(OUTPUT / f'official_cm_{variant}.csv', records[variant]['official']['confusion_matrix'])
        save_cm(OUTPUT / f'handmade_cm_{variant}.csv', records[variant]['handmade']['confusion_matrix'])
        pred = predictions[variant]['handmade']
        for i in range(len(yhm)):
            predict_rows.append({
                'variant': variant, 'writer': int(writers[i]), 'index': int(indices[i]),
                'true': LABELS[int(yhm[i])], 'predicted': LABELS[int(pred[i])],
                'correct': int(pred[i] == yhm[i])
            })
    save_csv(OUTPUT / 'handmade_predictions.csv', predict_rows)
    transitions = {
        variant: {
            'official': paired_stats(yva, baseline_official, predictions[variant]['official']),
            'handmade': paired_stats(yhm, baseline_handmade, predictions[variant]['handmade']),
        } for variant in ('size_only', 'stroke_only', 'size_plus_stroke')
    }
    summary = {
        'experiment': 'EXP-007 KNN Normal train-only size/stroke augmentation',
        'reference': 'EXP-005 CoM k=3 distance',
        'official_dataset_sha256': dataset_sha,
        'handmade_n': len(yhm), 'official_train_n': len(ytr), 'official_validation_n': len(yva),
        'random_seed': SEED, 'augmentation_fraction': AUG_FRACTION,
        'augmentation_design': 'Append 20% of class-stratified official train rows; no validation augmentation',
        'target_long_sides': list(TARGET_LONG_SIDES), 'target_size_counts': size_counts,
        'stroke_strength': STROKE_STRENGTH, 'stroke_method': 'cross-kernel gray erosion/dilation blend',
        'stroke_topology': 'preserve connected components and hole counts at threshold 128',
        'stroke_only_stats': stroke_stats, 'size_plus_stroke_stats': both_stats,
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'metric': 'minkowski', 'p': 2},
        'centering': 'MLP Normal CoM, threshold=32, integer shift zero-fill',
        'guardrail_min_official_accuracy': OFFICIAL_GUARDRAIL,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
        'candidates': rows, 'paired_vs_com': transitions,
        'selected_candidate_development_only': selected_candidate,
        'note': 'Hand-made is development-only; Original Test and final Test untouched.',
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nPAIRED vs CoM:')
    for name, val in transitions.items():
        print(name, val, flush=True)
    print('\nSELECTED DEVELOPMENT CANDIDATE:',
          selected_candidate['variant'] if selected_candidate else 'NONE', flush=True)
    if selected_candidate:
        print(f"Official={selected_candidate['official_accuracy']:.6f}, "
              f"Hand-made={selected_candidate['handmade_accuracy']:.6f}, "
              f"MacroF1={selected_candidate['handmade_macro_f1']:.6f}", flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP008_KNN_Normal_Aug_Ratio

원본: [05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py)  
Git blob: `22840a56aa789dd0b417bd191accb679f11b8c7e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-008: CoM KNN fixed k=3 distance; training-only augmentation-ratio comparison.

Place beside EXP004_KNN_Normal_CoM.py,
EXP006_KNN_Normal_BB_Size_fixed.py, and EXP007_KNN_Normal_Augmentation.py.
From repository root run:
    python 05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-008: CoM KNN fixed k=3 distance; training-only augmentation-ratio comparison.

Place beside EXP004_KNN_Normal_CoM.py,
EXP006_KNN_Normal_BB_Size_fixed.py, and EXP007_KNN_Normal_Augmentation.py.
From repository root run:
    python 05_model_optimize/KNN/Normal/scripts/EXP008_KNN_Normal_Aug_Ratio.py
"""
from __future__ import annotations

import csv
import json
import sys
import time
import platform
from pathlib import Path

import numpy as np
import sklearn

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    knn_model, sample_balanced, size_jitter, stroke_jitter,
    test_augmentation, save_csv, save_cm,
    EXPECTED_OFFICIAL_CORRECT, EXPECTED_HANDMADE_CORRECT,
    OFFICIAL_GUARDRAIL, SEED, TARGET_LONG_SIDES, STROKE_STRENGTH,
)

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP008_aug_ratio'
FRACTIONS = (0.10, 0.20, 0.30)


def main():
    test_augmentation()
    xtr, ytr, xva, yva, digest = fixed_split()
    xhm, yhm, writer_ids, sample_ids = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise ValueError('Unexpected dataset sizes')
    com = BatchedCoM()
    xt = com.transform(xtr)
    xv = com.transform(xva)
    xh = com.transform(xhm)
    print('Training=76800 / Official Validation=19200 / Hand-made=2560', flush=True)

    configurations = [('control_com', 0.0, 'none')]
    for fraction in FRACTIONS:
        configurations.append((f'size_{int(fraction * 100):02d}', fraction, 'size'))
    for fraction in FRACTIONS:
        configurations.append((f'combined_{int(fraction * 100):02d}', fraction, 'combined'))

    candidate_rows = []
    writer_rows = []
    class_rows = []
    prediction_rows = []
    details = {}
    predictions = {}
    control_official = control_handmade = None
    stats_by_fraction = {}

    for name, fraction, kind in configurations:
        print(f'\n[{name}] Preparing training rows...', flush=True)
        if kind == 'none':
            xx, yy = xt, ytr
            augmentation_n = 0
        else:
            idx = sample_balanced(ytr, fraction=fraction, seed=SEED)
            # Use the same fixed size choices and deterministic random seed as EXP-007.
            size_values, counts = size_jitter(xtr, idx, seed=SEED)
            stats_by_fraction[f'{kind}_{fraction}'] = {'sizes': counts}
            if kind == 'combined':
                aug, stroke_stats = stroke_jitter(size_values, strength=STROKE_STRENGTH)
                stats_by_fraction[f'{kind}_{fraction}']['stroke'] = stroke_stats
            else:
                aug = size_values
            xx = np.concatenate([xt, com.transform(aug)], axis=0)
            yy = np.concatenate([ytr, ytr[idx]], axis=0)
            augmentation_n = len(idx)
        model = knn_model()
        start = time.perf_counter()
        model.fit(xx, yy)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        op = model.predict(xv).astype(np.int64)
        official_seconds = time.perf_counter() - start
        start = time.perf_counter()
        hp = model.predict(xh).astype(np.int64)
        handmade_seconds = time.perf_counter() - start
        official = scores(yva, op)
        handmade = scores(yhm, hp)
        official_correct = int(np.sum(op == yva))
        handmade_correct = int(np.sum(hp == yhm))
        if name == 'control_com':
            if official_correct != EXPECTED_OFFICIAL_CORRECT or handmade_correct != EXPECTED_HANDMADE_CORRECT:
                raise RuntimeError('EXP-005 control mismatch; stop and inspect environment')
            control_official, control_handmade = op.copy(), hp.copy()
            print('EXP-005 control reproduced exactly', flush=True)
        if name in ('size_20', 'combined_20'):
            expected = {'size_20': (18470, 2110), 'combined_20': (18401, 2123)}[name]
            if (official_correct, handmade_correct) != expected:
                raise RuntimeError(f'EXP-007 {name} replication mismatch: {(official_correct, handmade_correct)} vs {expected}')
            print('EXP-007 20% reference reproduced exactly', flush=True)
        pass_guard = official['accuracy'] >= OFFICIAL_GUARDRAIL
        row = {
            'variant': name, 'augmentation_fraction': fraction, 'kind': kind,
            'train_n': len(yy), 'augmented_n': augmentation_n,
            'official_correct': official_correct, 'official_accuracy': official['accuracy'],
            'official_macro_f1': official['macro_f1'],
            'handmade_correct': handmade_correct, 'handmade_accuracy': handmade['accuracy'],
            'handmade_macro_f1': handmade['macro_f1'],
            'official_guardrail_pass': int(pass_guard),
            'official_delta_pp_vs_control': 100 * (official['accuracy'] - EXPECTED_OFFICIAL_CORRECT / 19200),
            'handmade_delta_pp_vs_control': 100 * (handmade['accuracy'] - EXPECTED_HANDMADE_CORRECT / 2560),
            'fit_seconds': fit_seconds, 'official_predict_seconds': official_seconds,
            'handmade_predict_seconds': handmade_seconds,
        }
        candidate_rows.append(row)
        details[name] = {
            'official': {'accuracy': official['accuracy'], 'macro_f1': official['macro_f1'],
                         'confusion_matrix': official['confusion_matrix']},
            'handmade': {'accuracy': handmade['accuracy'], 'macro_f1': handmade['macro_f1'],
                         'confusion_matrix': handmade['confusion_matrix']},
        }
        predictions[name] = (op, hp)
        for w in (41, 42, 43, 44):
            mask = writer_ids == w
            sc = scores(yhm[mask], hp[mask])
            writer_rows.append({'variant': name, 'writer': w, 'n': int(mask.sum()),
                                'accuracy': sc['accuracy'], 'macro_f1': sc['macro_f1']})
        for label_name in LABELS:
            p = handmade['per_class'][label_name]
            class_rows.append({'variant': name, 'class': label_name, 'support': p['support'],
                               'precision': p['precision'], 'recall': p['recall'], 'f1': p['f1']})
        print(f"{name:<15} train={len(yy)} | Official={official['accuracy']:.6f} "
              f"F1={official['macro_f1']:.6f} | Hand-made={handmade['accuracy']:.6f} "
              f"F1={handmade['macro_f1']:.6f} | guardrail={'PASS' if pass_guard else 'FAIL'}", flush=True)
        del model, xx, yy

    valid = [row for row in candidate_rows if row['official_guardrail_pass']]
    winner = max(valid, key=lambda row: (row['handmade_macro_f1'], row['handmade_accuracy'],
                                        row['official_macro_f1'])) if valid else None
    OUTPUT.mkdir(parents=True, exist_ok=True)
    save_csv(OUTPUT / 'candidate_results.csv', candidate_rows)
    save_csv(OUTPUT / 'by_writer.csv', writer_rows)
    save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
    for name in predictions:
        op, hp = predictions[name]
        save_cm(OUTPUT / f'official_cm_{name}.csv', details[name]['official']['confusion_matrix'])
        save_cm(OUTPUT / f'handmade_cm_{name}.csv', details[name]['handmade']['confusion_matrix'])
        for i, pred in enumerate(hp):
            prediction_rows.append({'variant': name, 'writer': int(writer_ids[i]),
                                    'index': int(sample_ids[i]), 'true': LABELS[int(yhm[i])],
                                    'predicted': LABELS[int(pred)], 'correct': int(pred == yhm[i])})
    save_csv(OUTPUT / 'handmade_predictions.csv', prediction_rows)
    paired = {
        name: {
            'official': paired_stats(yva, control_official, predictions[name][0]),
            'handmade': paired_stats(yhm, control_handmade, predictions[name][1]),
        } for name in predictions if name != 'control_com'
    }
    result = {
        'experiment': 'EXP-008 KNN Normal augmentation fraction search',
        'reference': 'EXP-005 CoM k=3 distance', 'source_augmentation': 'EXP007_KNN_Normal_Augmentation.py',
        'official_dataset_sha256': digest, 'train_n': len(ytr), 'validation_n': len(yva),
        'handmade_n': len(yhm), 'seed': SEED, 'fractions': list(FRACTIONS),
        'target_long_sides': list(TARGET_LONG_SIDES), 'stroke_strength': STROKE_STRENGTH,
        'guardrail_official_accuracy': OFFICIAL_GUARDRAIL,
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'metric': 'minkowski', 'p': 2},
        'candidates': candidate_rows, 'selected_development_candidate': winner,
        'pairwise_vs_control': paired, 'augmentation_stats': stats_by_fraction,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
        'note': 'Training-only augmentation. Hand-made used for model development comparison; not final holdout. Original Test and final Test untouched.',
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', winner['variant'] if winner else 'NONE', flush=True)
    if winner:
        print(f"Official={winner['official_accuracy']:.6f}, Hand-made={winner['handmade_accuracy']:.6f}, "
              f"MacroF1={winner['handmade_macro_f1']:.6f}", flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP009_KNN_Normal_Stroke_Strength

원본: [05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py)  
Git blob: `c50900f37f3a50994dedc8edd6906b685db5907f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-009: training-only stroke intensity search with size augmentation fixed at 30%.

Place beside EXP004_KNN_Normal_CoM.py,
EXP006_KNN_Normal_BB_Size_fixed.py, and EXP007_KNN_Normal_Augmentation.py.
From repository root run:
    python 05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-009: training-only stroke intensity search with size augmentation fixed at 30%.

Place beside EXP004_KNN_Normal_CoM.py,
EXP006_KNN_Normal_BB_Size_fixed.py, and EXP007_KNN_Normal_Augmentation.py.
From repository root run:
    python 05_model_optimize/KNN/Normal/scripts/EXP009_KNN_Normal_Stroke_Strength.py
"""
from __future__ import annotations

import csv
import json
import sys
import time
import platform
from pathlib import Path

import numpy as np
import sklearn

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    knn_model, sample_balanced, size_jitter, stroke_jitter,
    test_augmentation, save_csv, save_cm,
    EXPECTED_OFFICIAL_CORRECT, EXPECTED_HANDMADE_CORRECT,
    OFFICIAL_GUARDRAIL, SEED, TARGET_LONG_SIDES, STROKE_STRENGTH,
)

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP009_stroke_strength'
AUG_FRACTION = 0.30
STRENGTHS = (0.12, 0.20, 0.28, 0.36, 0.44)


def main():
    test_augmentation()
    xtr, ytr, xva, yva, digest = fixed_split()
    xhm, yhm, writer_ids, sample_ids = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise ValueError('Unexpected dataset sizes')
    com = BatchedCoM()
    xt = com.transform(xtr)
    xv = com.transform(xva)
    xh = com.transform(xhm)
    print('Training=76800 / Official Validation=19200 / Hand-made=2560', flush=True)

    configurations = [('control_com', None), ('size_30', 0.0)]
    configurations += [(f'combined_strength_{int(round(strength * 100)):02d}', strength)
                       for strength in STRENGTHS]
    idx = sample_balanced(ytr, fraction=AUG_FRACTION, seed=SEED)
    size_values, size_counts = size_jitter(xtr, idx, seed=SEED)
    augmented_n = len(idx)
    print(f'Augmentation fixed at {AUG_FRACTION:.0%}: {augmented_n} training-only rows', flush=True)

    candidate_rows = []
    writer_rows = []
    class_rows = []
    prediction_rows = []
    details = {}
    predictions = {}
    control_official = control_handmade = None
    stats_by_strength = {}

    for name, strength in configurations:
        print(f'\n[{name}] Preparing training rows...', flush=True)
        if strength is None:
            xx, yy = xt, ytr
            augmentation_n = 0
        else:
            if strength == 0:
                aug = size_values
                stroke_stats = {'changed': 0, 'topology_rejected': 0}
            else:
                aug, stroke_stats = stroke_jitter(size_values, strength=strength)
            stats_by_strength[name] = {'size_counts': size_counts, 'stroke': stroke_stats}
            xx = np.concatenate([xt, com.transform(aug)], axis=0)
            yy = np.concatenate([ytr, ytr[idx]], axis=0)
            augmentation_n = len(idx)
        model = knn_model()
        start = time.perf_counter()
        model.fit(xx, yy)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        op = model.predict(xv).astype(np.int64)
        official_seconds = time.perf_counter() - start
        start = time.perf_counter()
        hp = model.predict(xh).astype(np.int64)
        handmade_seconds = time.perf_counter() - start
        official = scores(yva, op)
        handmade = scores(yhm, hp)
        official_correct = int(np.sum(op == yva))
        handmade_correct = int(np.sum(hp == yhm))
        if name == 'control_com':
            if official_correct != EXPECTED_OFFICIAL_CORRECT or handmade_correct != EXPECTED_HANDMADE_CORRECT:
                raise RuntimeError('EXP-005 control mismatch; stop and inspect environment')
            control_official, control_handmade = op.copy(), hp.copy()
            print('EXP-005 control reproduced exactly', flush=True)
        if name == 'combined_strength_28':
            expected = (18395, 2127)  # EXP-008 combined_30: official and Hand-made counts
            if (official_correct, handmade_correct) != expected:
                raise RuntimeError(f'EXP-008 combined_30 reference mismatch: {(official_correct, handmade_correct)} vs {expected}')
            print('EXP-008 combined_30 reference reproduced exactly', flush=True)
        pass_guard = official['accuracy'] >= OFFICIAL_GUARDRAIL
        row = {
            'variant': name, 'augmentation_fraction': 0.0 if strength is None else AUG_FRACTION,
            'stroke_strength': strength,
            'train_n': len(yy), 'augmented_n': augmentation_n,
            'official_correct': official_correct, 'official_accuracy': official['accuracy'],
            'official_macro_f1': official['macro_f1'],
            'handmade_correct': handmade_correct, 'handmade_accuracy': handmade['accuracy'],
            'handmade_macro_f1': handmade['macro_f1'],
            'official_guardrail_pass': int(pass_guard),
            'official_delta_pp_vs_control': 100 * (official['accuracy'] - EXPECTED_OFFICIAL_CORRECT / 19200),
            'handmade_delta_pp_vs_control': 100 * (handmade['accuracy'] - EXPECTED_HANDMADE_CORRECT / 2560),
            'fit_seconds': fit_seconds, 'official_predict_seconds': official_seconds,
            'handmade_predict_seconds': handmade_seconds,
        }
        candidate_rows.append(row)
        details[name] = {
            'official': {'accuracy': official['accuracy'], 'macro_f1': official['macro_f1'],
                         'confusion_matrix': official['confusion_matrix']},
            'handmade': {'accuracy': handmade['accuracy'], 'macro_f1': handmade['macro_f1'],
                         'confusion_matrix': handmade['confusion_matrix']},
        }
        predictions[name] = (op, hp)
        for w in (41, 42, 43, 44):
            mask = writer_ids == w
            sc = scores(yhm[mask], hp[mask])
            writer_rows.append({'variant': name, 'writer': w, 'n': int(mask.sum()),
                                'accuracy': sc['accuracy'], 'macro_f1': sc['macro_f1']})
        for label_name in LABELS:
            p = handmade['per_class'][label_name]
            class_rows.append({'variant': name, 'class': label_name, 'support': p['support'],
                               'precision': p['precision'], 'recall': p['recall'], 'f1': p['f1']})
        print(f"{name:<15} train={len(yy)} | Official={official['accuracy']:.6f} "
              f"F1={official['macro_f1']:.6f} | Hand-made={handmade['accuracy']:.6f} "
              f"F1={handmade['macro_f1']:.6f} | guardrail={'PASS' if pass_guard else 'FAIL'}", flush=True)
        del model, xx, yy

    valid = [row for row in candidate_rows if row['official_guardrail_pass']]
    winner = max(valid, key=lambda row: (row['handmade_macro_f1'], row['handmade_accuracy'],
                                        row['official_macro_f1'])) if valid else None
    OUTPUT.mkdir(parents=True, exist_ok=True)
    save_csv(OUTPUT / 'candidate_results.csv', candidate_rows)
    save_csv(OUTPUT / 'by_writer.csv', writer_rows)
    save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
    for name in predictions:
        op, hp = predictions[name]
        save_cm(OUTPUT / f'official_cm_{name}.csv', details[name]['official']['confusion_matrix'])
        save_cm(OUTPUT / f'handmade_cm_{name}.csv', details[name]['handmade']['confusion_matrix'])
        for i, pred in enumerate(hp):
            prediction_rows.append({'variant': name, 'writer': int(writer_ids[i]),
                                    'index': int(sample_ids[i]), 'true': LABELS[int(yhm[i])],
                                    'predicted': LABELS[int(pred)], 'correct': int(pred == yhm[i])})
    save_csv(OUTPUT / 'handmade_predictions.csv', prediction_rows)
    paired = {
        name: {
            'official': paired_stats(yva, predictions['combined_strength_28'][0], predictions[name][0]),
            'handmade': paired_stats(yhm, predictions['combined_strength_28'][1], predictions[name][1]),
        } for name in predictions if name != 'combined_strength_28'
    }
    result = {
        'experiment': 'EXP-009 KNN Normal stroke strength comparison',
        'reference': 'EXP-008 combined_30 stroke strength 0.28; EXP-005 CoM k=3 distance',
        'source_augmentation': 'EXP007_KNN_Normal_Augmentation.py',
        'official_dataset_sha256': digest, 'train_n': len(ytr), 'validation_n': len(yva),
        'handmade_n': len(yhm), 'seed': SEED, 'augmentation_fraction': AUG_FRACTION,
        'tested_stroke_strengths': list(STRENGTHS),
        'target_long_sides': list(TARGET_LONG_SIDES), 'reference_stroke_strength': STROKE_STRENGTH,
        'guardrail_official_accuracy': OFFICIAL_GUARDRAIL,
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'metric': 'minkowski', 'p': 2},
        'candidates': candidate_rows, 'selected_development_candidate': winner,
        'pairwise_vs_combined_30': paired, 'augmentation_stats': stats_by_strength,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
        'note': 'Training-only augmentation. Fixed original training and balanced size augmentation at 30%. Only stroke blend strength changes among combined candidates. Hand-made is development data, not independent final test. Original Test and final Test untouched.',
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', winner['variant'] if winner else 'NONE', flush=True)
    if winner:
        print(f"Official={winner['official_accuracy']:.6f}, Hand-made={winner['handmade_accuracy']:.6f}, "
              f"MacroF1={winner['handmade_macro_f1']:.6f}", flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP014_KNN_Normal_Targeted_Augmentation

원본: [05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py)  
Git blob: `705f71b31cb9a0e814cd254d1e67642f66a7d9e9`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-014: class-targeted official-training augmentation for KNN Normal.

Run from repository root:
python 05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py

Requires EXP004, EXP006_fixed, EXP007, EXP011, EXP012, EXP013 scripts
and EXP013_hog_mix_ratio/handmade_predictions_pixel_20_hog_80.csv.
Hand-made is used ONLY to select weak classes and evaluate, NEVER to fit.
Such repeated model selection can overfit the Hand-made development set.
The official Test is never loa



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-014: class-targeted official-training augmentation for KNN Normal.

Run from repository root:
python 05_model_optimize/KNN/Normal/scripts/EXP014_KNN_Normal_Targeted_Augmentation.py

Requires EXP004, EXP006_fixed, EXP007, EXP011, EXP012, EXP013 scripts
and EXP013_hog_mix_ratio/handmade_predictions_pixel_20_hog_80.csv.
Hand-made is used ONLY to select weak classes and evaluate, NEVER to fit.
Such repeated model selection can overfit the Hand-made development set.
The official Test is never loaded.
"""
from __future__ import annotations

import csv
import json
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import LABELS, fixed_split, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    sample_balanced, size_jitter, stroke_jitter, test_augmentation,
    save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, hybrid_features, test_hog

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP014_targeted_augmentation'
PRED_CSV = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP013_hog_mix_ratio' / 'handmade_predictions_pixel_20_hog_80.csv'
BASE_FRACTION = 0.30
STROKE_STRENGTH = 0.44
PIXEL_WEIGHT = 0.20
N_WEAK_CLASSES = 4
EXTRA_COUNTS = (3840, 7680)   # +5%, +10% of original official train (76,800)
CONTROL_CORRECT = (18851, 2360)  # exact EXP-013 20:80 baseline


def weak_classes_from_dev_predictions(path, y_hm, writers, indices):
    if not path.is_file():
        raise FileNotFoundError(f'EXP-013 prediction file not found: {path}')
    with path.open('r', newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    if len(rows) != len(y_hm):
        raise RuntimeError(f'EXP-013 prediction count mismatch: {len(rows)} != {len(y_hm)}')
    counts = Counter()
    pairs = Counter()
    correct = 0
    for i, row in enumerate(rows):
        true = LABELS[int(y_hm[i])]
        if (int(row['writer']) != int(writers[i]) or
                int(row['index']) != int(indices[i]) or
                row['true'] != true):
            raise RuntimeError(f'EXP-013 prediction order or true labels mismatch at row {i}')
        pred = row['predicted']
        if pred not in LABELS:
            raise RuntimeError(f'Unexpected prediction {pred}')
        if pred == true:
            correct += 1
        else:
            counts[true] += 1
            pairs[(true, pred)] += 1
    if correct != CONTROL_CORRECT[1]:
        raise RuntimeError(f'EXP-013 Hand-made prediction reference mismatch: {correct}')
    rank = sorted(range(16), key=lambda c: (-counts[LABELS[c]], c))
    selected = rank[:N_WEAK_CLASSES]
    if len(selected) != N_WEAK_CLASSES:
        raise RuntimeError('Insufficient targeted classes')
    return selected, counts, pairs


def select_extra(y, already_selected, total, target_classes, seed):
    """Choose additional official-training samples disjoint from the baseline aug indices."""
    rng = np.random.default_rng(seed)
    blocked = np.zeros(len(y), dtype=bool)
    blocked[already_selected] = True
    classes = list(target_classes)
    if total % len(classes):
        raise ValueError('Extra count must be divisible by target class count')
    per_class = total // len(classes)
    pieces = []
    for cls in classes:
        available = np.flatnonzero((y == cls) & ~blocked)
        if len(available) < per_class:
            raise RuntimeError(f'Insufficient unused training examples for {LABELS[cls]}: '
                               f'needed={per_class}, available={len(available)}')
        pieces.append(rng.choice(available, size=per_class, replace=False))
    chosen = np.sort(np.concatenate(pieces))
    if len(chosen) != total or len(np.unique(chosen)) != total or np.any(blocked[chosen]):
        raise RuntimeError('Additional sample selection check failed')
    return chosen


def augmented_features(x_train, indices, com, name):
    print(f'[{name}] Generating {len(indices)} official-training-only augmented rows...', flush=True)
    sized, size_stats = size_jitter(x_train, indices, seed=SEED + 140)
    strokes, stroke_stats = stroke_jitter(sized, strength=STROKE_STRENGTH)
    centered = com.transform(strokes)
    corrected, deskew_stats = deskew(centered)
    hog = hog_features(corrected)
    features = hybrid_features(corrected, hog, pixel_weight=PIXEL_WEIGHT)
    del sized, strokes, centered, corrected, hog
    return features, {'size': size_stats, 'stroke': stroke_stats, 'deskew': deskew_stats}


def evaluate(name, features_train, labels_train, val_features, y_val,
             hm_features, y_hm, writers, indices, records, rows, writer_rows,
             class_rows, baseline_predictions):
    print(f'\n[{name}] Fit / predict ({len(labels_train)} training rows)...', flush=True)
    knn = KNeighborsClassifier(n_neighbors=3, weights='distance', metric='euclidean',
                               algorithm='brute', n_jobs=-1)
    t = time.perf_counter()
    knn.fit(features_train, labels_train)
    fit_s = time.perf_counter() - t
    t = time.perf_counter()
    pred_val = np.asarray(knn.predict(val_features), dtype=np.int64)
    val_s = time.perf_counter() - t
    t = time.perf_counter()
    pred_hm = np.asarray(knn.predict(hm_features), dtype=np.int64)
    hm_s = time.perf_counter() - t
    official = scores(y_val, pred_val)
    handmade = scores(y_hm, pred_hm)
    correct = (int((pred_val == y_val).sum()), int((pred_hm == y_hm).sum()))
    if name == 'control':
        if correct != CONTROL_CORRECT:
            raise RuntimeError(f'EXP-013 20:80 control mismatch: {correct} != {CONTROL_CORRECT}')
        baseline_predictions['official'] = pred_val.copy()
        baseline_predictions['handmade'] = pred_hm.copy()
        print('EXP-013 20:80 baseline reproduced exactly', flush=True)
    record = {
        'variant': name, 'train_n': int(len(labels_train)),
        'extra_n': int(len(labels_train) - 99840),
        'official_correct': correct[0], 'official_accuracy': official['accuracy'],
        'official_macro_f1': official['macro_f1'],
        'handmade_correct': correct[1], 'handmade_accuracy': handmade['accuracy'],
        'handmade_macro_f1': handmade['macro_f1'],
        'official_guardrail_pass': int(official['accuracy'] >= OFFICIAL_GUARDRAIL),
        'fit_seconds': fit_s, 'official_predict_seconds': val_s,
        'handmade_predict_seconds': hm_s,
    }
    rows.append(record)
    records[name] = {'row': record, 'official': official, 'handmade': handmade}
    if name != 'control':
        records[name]['paired_vs_control'] = {
            'official': paired_stats(y_val, baseline_predictions['official'], pred_val),
            'handmade': paired_stats(y_hm, baseline_predictions['handmade'], pred_hm),
        }
    for w in (41, 42, 43, 44):
        m = writers == w
        score = scores(y_hm[m], pred_hm[m])
        writer_rows.append({'variant': name, 'writer': w, 'n': int(m.sum()),
                            'accuracy': score['accuracy'], 'macro_f1': score['macro_f1']})
    for c in LABELS:
        s = handmade['per_class'][c]
        class_rows.append({'variant': name, 'class': c, 'support': s['support'],
                           'precision': s['precision'], 'recall': s['recall'], 'f1': s['f1']})
    save_csv(OUT / 'candidate_results.csv', rows)
    save_csv(OUT / 'by_writer.csv', writer_rows)
    save_csv(OUT / 'per_class_handmade.csv', class_rows)
    save_cm(OUT / f'official_cm_{name}.csv', official['confusion_matrix'])
    save_cm(OUT / f'handmade_cm_{name}.csv', handmade['confusion_matrix'])
    save_csv(OUT / f'handmade_predictions_{name}.csv', [
        {'writer': int(writers[i]), 'index': int(indices[i]),
         'true': LABELS[int(y_hm[i])], 'predicted': LABELS[int(pred_hm[i])],
         'correct': int(pred_hm[i] == y_hm[i])} for i in range(len(y_hm))
    ])
    (OUT / 'metrics_partial.json').write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{name:<17} Official={official["accuracy"]:.6f} F1={official["macro_f1"]:.6f} | '
          f'Hand-made={handmade["accuracy"]:.6f} F1={handmade["macro_f1"]:.6f} | '
          f'guardrail={"PASS" if record["official_guardrail_pass"] else "FAIL"} | '
          f'predict_s={val_s:.1f}+{hm_s:.1f}', flush=True)
    del knn


def main():
    test_augmentation()
    test_deskew()
    test_hog()
    xtr, ytr, xva, yva, sha = fixed_split()
    xhm, yhm, writers, indices = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise RuntimeError('Unexpected dataset sizes')
    weak, error_counts, confused_pairs = weak_classes_from_dev_predictions(
        PRED_CSV, yhm, writers, indices)
    print('Weak classes selected from EXP-013 Hand-made DEVELOPMENT predictions:',
          [LABELS[c] for c in weak], flush=True)
    print('Errors in selected classes:', {LABELS[c]: error_counts[LABELS[c]] for c in weak}, flush=True)
    OUT.mkdir(parents=True, exist_ok=True)
    save_csv(OUT / 'selected_classes.csv', [
        {'class': c, 'handmade_errors': error_counts[c],
         'selected_for_targeted_augmentation': int(c in {LABELS[k] for k in weak})}
        for c in LABELS
    ])
    save_csv(OUT / 'top_confusions.csv', [
        {'true': pair[0], 'predicted': pair[1], 'count': count}
        for pair, count in sorted(confused_pairs.items(), key=lambda x: (-x[1], x[0]))
    ])
    selected = sample_balanced(ytr, fraction=BASE_FRACTION, seed=SEED)
    base_size, base_size_stats = size_jitter(xtr, selected, seed=SEED)
    base_extra, base_stroke_stats = stroke_jitter(base_size, strength=STROKE_STRENGTH)
    raw_base_train = np.concatenate([xtr, base_extra], axis=0)
    base_labels = np.concatenate([ytr, ytr[selected]], axis=0)
    if len(base_labels) != 99840:
        raise RuntimeError('EXP-013 baseline augmentation count mismatch')
    del base_size, base_extra
    com = BatchedCoM()
    t = time.perf_counter()
    base_images, _ = deskew(com.transform(raw_base_train))
    val_images, _ = deskew(com.transform(xva))
    hand_images, _ = deskew(com.transform(xhm))
    del raw_base_train
    print(f'Preprocessing done in {time.perf_counter()-t:.1f}s', flush=True)
    t = time.perf_counter()
    base_feats = hybrid_features(base_images, hog_features(base_images), pixel_weight=PIXEL_WEIGHT)
    val_feats = hybrid_features(val_images, hog_features(val_images), pixel_weight=PIXEL_WEIGHT)
    hm_feats = hybrid_features(hand_images, hog_features(hand_images), pixel_weight=PIXEL_WEIGHT)
    del base_images, val_images, hand_images
    print(f'Pixel:HOG 20:80 features done in {time.perf_counter()-t:.1f}s', flush=True)
    rows, writer_rows, class_rows, records, control_preds = [], [], [], {}, {}
    evaluate('control', base_feats, base_labels, val_feats, yva, hm_feats, yhm,
             writers, indices, records, rows, writer_rows, class_rows, control_preds)

    variants = [
        ('uniform_5', EXTRA_COUNTS[0], list(range(16)), SEED + 141),
        ('targeted_5', EXTRA_COUNTS[0], weak, SEED + 142),
        ('targeted_10', EXTRA_COUNTS[1], weak, SEED + 142),
    ]
    selection_records = {}
    for name, n, classes, seed in variants:
        ix = select_extra(ytr, selected, n, classes, seed=seed)
        extras, stats = augmented_features(xtr, ix, com, name)
        combined_features = np.concatenate([base_feats, extras], axis=0)
        combined_labels = np.concatenate([base_labels, ytr[ix]], axis=0)
        selection_records[name] = {
            'added_n': int(len(ix)), 'classes': [LABELS[k] for k in classes],
            'counts': {LABELS[k]: int(np.sum(ytr[ix] == k)) for k in classes},
            'stats': stats,
        }
        evaluate(name, combined_features, combined_labels, val_feats, yva,
                 hm_feats, yhm, writers, indices, records, rows, writer_rows,
                 class_rows, control_preds)
        del extras, combined_features, combined_labels

    eligible = [r for r in rows if r['official_guardrail_pass']]
    winner = max(eligible, key=lambda r: (r['handmade_macro_f1'],
                                         r['handmade_accuracy'], r['official_accuracy']))
    summary = {
        'experiment': 'EXP-014', 'dataset_sha256': sha,
        'official_train_n': len(ytr), 'official_validation_n': len(yva),
        'handmade_development_n': len(yhm), 'baseline_train_n': len(base_labels),
        'baseline_aug_fraction': BASE_FRACTION, 'stroke_strength': STROKE_STRENGTH,
        'base_size_stats': base_size_stats, 'base_stroke_stats': base_stroke_stats,
        'knn': {'k': 3, 'weights': 'distance', 'metric': 'euclidean'},
        'preprocessing': 'CoM -> Deskew', 'pixel_weight': PIXEL_WEIGHT,
        'weak_classes': [LABELS[c] for c in weak],
        'selection_source': str(PRED_CSV.relative_to(ROOT)),
        'targeting_counts': {LABELS[c]: error_counts[LABELS[c]] for c in weak},
        'additional_augmentation': selection_records,
        'candidates': rows, 'selected_development_candidate': winner,
        'results': records,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
        'note': 'Target classes chosen from repeatedly used Hand-made development labels. '
                'Only official Training images were fitted/augmented. Do not claim '
                'unbiased generalization. Official Test not accessed.',
    }
    (OUT / 'metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', winner['variant'], flush=True)
    print('Saved:', OUT, flush=True)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")